# 02. Grid Search

Searches for the best hyperparameters and blend weights for each endpoint. It loads the processed features that `01_data_exploration.ipynb` already wrote to disk, and it does not rerun `build_patient_features`.

It runs in order. First, a bounded per endpoint XGB hyperparameter search (`search_xgb_hyperparams`) and a Coxnet l1_ratio search (`search_coxnet_l1_ratio`), both defined in `liverrisk/models.py`. Then, the per endpoint blend weight search (`search_blend_weights`, in `liverrisk/blend.py`), which uses the XGB hyperparameters just tuned above (through `config.xgb_hyperparams_hep()` and `_death()`, freshly written to disk by the first step) rather than whatever was sitting in `best_config.json` on checkout.

Each search writes its winner back into `liverrisk/best_config.json` through `config.update_config(...)`. `xgb_hyperparams_hep`, `xgb_hyperparams_death`, and `coxnet_hyperparams` (with per endpoint `l1_ratio_hep`/`l1_ratio_death`) are all stored separately for each endpoint, just like `blend_weights_hep`/`blend_weights_death`; there is no shared fallback for either. `coxnet_alpha_search` is exposed in `liverrisk/config.py` but isn't searched here, since alpha is already tuned per fit by `fit_coxnet_with_alpha_cv`.

In [8]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "liverrisk" / "features.py").exists():
            return p
    raise RuntimeError("Could not locate repo root (liverrisk/features.py not found)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: c:\Users\paabl\OneDrive\Documents\GitHub\TFG_pabloCalderon


In [9]:
import numpy as np
import pandas as pd

from liverrisk import config
from liverrisk.blend import search_blend_weights
from liverrisk.cv import cv_cindex_blend, cv_cindex_coxnet, cv_cindex_rsf, cv_cindex_xgb
from liverrisk.features import load_features
from liverrisk.models import (
    HAS_XGB,
    search_coxnet_l1_ratio,
    search_xgb_hyperparams,
)

PROCESSED_DIR = REPO_ROOT / "liverrisk" / "data" / "processed"

# Reduce these for a fast verification run; bump back up for a trustworthy
# final search. n_points controls the blend-weight grid density (n_points=6
# matches the original notebook); n_repeats_report controls how many times
# the OLD-vs-NEW comparison at the bottom is repeated. N_XGB_CANDIDATES
# controls how many random XGB hyperparameter combinations get tried per
# endpoint in the hyperparameter search section below.
N_POINTS = 6
N_REPEATS_REPORT = 3
N_XGB_CANDIDATES = 12

X_hep, y_hep, hep_event, hep_time = load_features("hep", PROCESSED_DIR)
X_death, y_death, death_event, death_time = load_features("death", PROCESSED_DIR)

print(f"hepatic: n={len(X_hep)}, events={hep_event.sum()} ({hep_event.mean():.3%})")
print(f"death  : n={len(X_death)}, events={death_event.sum()} ({death_event.mean():.3%})")

hepatic: n=1253, events=47 (3.751%)
death  : n=984, events=76 (7.724%)


## XGB hyperparameter search (per endpoint)

`search_xgb_hyperparams` randomly samples `N_XGB_CANDIDATES` combinations from a bounded grid over learning_rate, max_depth, n_estimators, min_child_weight, subsample, colsample_bytree, reg_lambda, and reg_alpha, scoring each one with `cv_cindex_xgb` using a single CV repeat. It runs separately for the hepatic and death endpoints, since there's no reason to expect the two would share one good XGB configuration.

In [10]:
print("Searching XGB hyperparameters (hepatic)...")
hep_xgb_params, hep_xgb_search_mean, hep_xgb_search_df = search_xgb_hyperparams(
    X_hep, y_hep, hep_event, hep_time, n_candidates=N_XGB_CANDIDATES,
)
print(f"Best hepatic XGB params = {hep_xgb_params}, search-CV mean (1 repeat) = {hep_xgb_search_mean:.4f}")

print("\nSearching XGB hyperparameters (death)...")
death_xgb_params, death_xgb_search_mean, death_xgb_search_df = search_xgb_hyperparams(
    X_death, y_death, death_event, death_time, n_candidates=N_XGB_CANDIDATES,
)
print(f"Best death XGB params = {death_xgb_params}, search-CV mean (1 repeat) = {death_xgb_search_mean:.4f}")

Searching XGB hyperparameters (hepatic)...
Best hepatic XGB params = {'learning_rate': 0.025, 'max_depth': 3, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_lambda': 5.0, 'reg_alpha': 0.5}, search-CV mean (1 repeat) = 0.7402

Searching XGB hyperparameters (death)...
Best death XGB params = {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 0.85, 'reg_lambda': 1.0, 'reg_alpha': 0.5}, search-CV mean (1 repeat) = 0.9366


## Coxnet l1_ratio search (per endpoint)

`search_coxnet_l1_ratio` sweeps `l1_ratio` over the values `[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]` and scores each one with `cv_cindex_coxnet`. Alpha is left untouched here, since it's already tuned per fit by `fit_coxnet_with_alpha_cv`.

In [11]:
print("Searching Coxnet l1_ratio (hepatic)...")
hep_l1_ratio, hep_cox_search_mean, hep_cox_search_df = search_coxnet_l1_ratio(X_hep, y_hep, hep_event, hep_time)
print(f"Best hepatic l1_ratio = {hep_l1_ratio}, search-CV mean (1 repeat) = {hep_cox_search_mean:.4f}")

print("\nSearching Coxnet l1_ratio (death)...")
death_l1_ratio, death_cox_search_mean, death_cox_search_df = search_coxnet_l1_ratio(X_death, y_death, death_event, death_time)
print(f"Best death l1_ratio = {death_l1_ratio}, search-CV mean (1 repeat) = {death_cox_search_mean:.4f}")

Searching Coxnet l1_ratio (hepatic)...
Best hepatic l1_ratio = 0.1, search-CV mean (1 repeat) = 0.7665

Searching Coxnet l1_ratio (death)...
Best death l1_ratio = 0.99, search-CV mean (1 repeat) = 0.9404


## Save the tuned hyperparameters to `best_config.json`

`config.update_config` merges the XGB hyperparameters and Coxnet l1_ratio values into the JSON config file, without touching `blend_weights_hep`/`blend_weights_death` or `coxnet_alpha_search`. Writing these now, before the blend search runs, is what lets the blend search below read the freshly tuned `config.xgb_hyperparams_hep()`/`_death()` instead of whatever defaults were in the file on checkout.

In [12]:
config.update_config(
    xgb_hyperparams_hep=dict(hep_xgb_params),
    xgb_hyperparams_death=dict(death_xgb_params),
    coxnet_hyperparams={"l1_ratio_hep": hep_l1_ratio, "l1_ratio_death": death_l1_ratio},
)
print("Updated liverrisk/best_config.json:")
print(config.get_config())

Updated liverrisk/best_config.json:
{'xgb_hyperparams_hep': {'learning_rate': 0.025, 'max_depth': 3, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_lambda': 5.0, 'reg_alpha': 0.5}, 'xgb_hyperparams_death': {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 0.85, 'reg_lambda': 1.0, 'reg_alpha': 0.5}, 'coxnet_alpha_search': {'n_alphas': 30, 'n_splits': 3}, 'coxnet_hyperparams': {'l1_ratio_hep': 0.1, 'l1_ratio_death': 0.99}, 'blend_weights_hep': [0.0, 1.0, 0.0], 'blend_weights_death': [0.4, 0.4, 0.2]}


## Blend weight search (per endpoint)

`search_blend_weights` grid searches over `(w_cox, w_rsf, w_xgb)` triples that sum to 1, including the edge cases where one or two of the weights are 0.

In [13]:
print("Searching blend weights (hepatic)...")
hep_weights, hep_search_mean, hep_search_df = search_blend_weights(
    X_hep, y_hep, hep_event, hep_time, n_points=N_POINTS, xgb_params=config.xgb_hyperparams_hep(),
)
print(f"Best hepatic weights (w_cox, w_rsf, w_xgb) = {hep_weights}, search-CV mean (1 repeat) = {hep_search_mean:.4f}")

print("\nSearching blend weights (death)...")
death_weights, death_search_mean, death_search_df = search_blend_weights(
    X_death, y_death, death_event, death_time, n_points=N_POINTS, xgb_params=config.xgb_hyperparams_death(),
)
print(f"Best death weights (w_cox, w_rsf, w_xgb) = {death_weights}, search-CV mean (1 repeat) = {death_search_mean:.4f}")

Searching blend weights (hepatic)...
Best hepatic weights (w_cox, w_rsf, w_xgb) = (0.0, 1.0, 0.0), search-CV mean (1 repeat) = 0.7998

Searching blend weights (death)...
Best death weights (w_cox, w_rsf, w_xgb) = (0.4, 0.4, 0.2), search-CV mean (1 repeat) = 0.9592


In [14]:
hep_search_df.head(10)

,w_cox,w_rsf,w_xgb,mean,std
0,0.0,1.0,0.0,0.799810,0.104174
1,0.2,0.8,0.0,0.798936,0.093203
2,0.4,0.6,0.0,0.793418,0.081944
3,0.0,0.8,0.2,0.790036,0.101068
4,0.2,0.6,0.2,0.789966,0.091892
5,0.0,0.6,0.4,0.782778,0.098876
6,0.4,0.4,0.2,0.778813,0.078959
7,0.6,0.4,0.0,0.776820,0.082331
8,0.2,0.4,0.4,0.776142,0.087822
9,0.0,0.4,0.6,0.767956,0.097623


In [15]:
death_search_df.head(10)

,w_cox,w_rsf,w_xgb,mean,std
0,0.4,0.4,0.2,0.959158,0.008444
1,0.4,0.6,0.0,0.957298,0.007443
2,0.2,0.6,0.2,0.956855,0.005809
3,0.6,0.4,0.0,0.955975,0.009779
4,0.2,0.8,0.0,0.955714,0.007238
5,0.4,0.2,0.4,0.955651,0.012478
6,0.6,0.2,0.2,0.955296,0.012099
7,0.2,0.4,0.4,0.954809,0.006645
8,0.2,0.2,0.6,0.952657,0.013047
9,0.6,0.0,0.4,0.952187,0.013336


## Write the winning blend weights to `best_config.json`

`config.update_config` merges `blend_weights_hep` and `blend_weights_death` into the JSON config file, leaving the XGB hyperparameters, Coxnet l1_ratio, and `coxnet_alpha_search` untouched, since those were already written by the hyperparameter search earlier in the notebook.

In [16]:
config.update_config(
    blend_weights_hep=list(hep_weights),
    blend_weights_death=list(death_weights),
)
print("Updated liverrisk/best_config.json:")
print(config.get_config())

Updated liverrisk/best_config.json:
{'xgb_hyperparams_hep': {'learning_rate': 0.025, 'max_depth': 3, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_lambda': 5.0, 'reg_alpha': 0.5}, 'xgb_hyperparams_death': {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 0.85, 'reg_lambda': 1.0, 'reg_alpha': 0.5}, 'coxnet_alpha_search': {'n_alphas': 30, 'n_splits': 3}, 'coxnet_hyperparams': {'l1_ratio_hep': 0.1, 'l1_ratio_death': 0.99}, 'blend_weights_hep': [0.0, 1.0, 0.0], 'blend_weights_death': [0.4, 0.4, 0.2]}


## Individual model CV

Runs cross validated C index scores for each model on its own (Coxnet, Random Survival Forest, and XGBoost) so they can be compared against the blended model below.

In [17]:
hep_cox_cv = cv_cindex_coxnet(X_hep, y_hep, hep_event, n_repeats=3, l1_ratio=config.coxnet_l1_ratio_hep())
death_cox_cv = cv_cindex_coxnet(X_death, y_death, death_event, n_repeats=3, l1_ratio=config.coxnet_l1_ratio_death())
print(f"Coxnet hepatic: mean={hep_cox_cv[0]:.4f}, std={hep_cox_cv[1]:.4f}")
print(f"Coxnet death  : mean={death_cox_cv[0]:.4f}, std={death_cox_cv[1]:.4f}")

hep_rsf_cv = cv_cindex_rsf(X_hep, y_hep, hep_event, n_repeats=3, n_estimators=150)
death_rsf_cv = cv_cindex_rsf(X_death, y_death, death_event, n_repeats=3, n_estimators=150)
print(f"RSF hepatic   : mean={hep_rsf_cv[0]:.4f}, std={hep_rsf_cv[1]:.4f}")
print(f"RSF death     : mean={death_rsf_cv[0]:.4f}, std={death_rsf_cv[1]:.4f}")

if HAS_XGB:
    hep_xgb_cv = cv_cindex_xgb(X_hep, hep_event, hep_time, n_repeats=3, xgb_params=config.xgb_hyperparams_hep())
    death_xgb_cv = cv_cindex_xgb(X_death, death_event, death_time, n_repeats=3, xgb_params=config.xgb_hyperparams_death())
    print(f"XGB hepatic   : mean={hep_xgb_cv[0]:.4f}, std={hep_xgb_cv[1]:.4f}")
    print(f"XGB death     : mean={death_xgb_cv[0]:.4f}, std={death_xgb_cv[1]:.4f}")
else:
    hep_xgb_cv = death_xgb_cv = (np.nan, np.nan)

Coxnet hepatic: mean=0.7656, std=0.0931
Coxnet death  : mean=0.9167, std=0.0383
RSF hepatic   : mean=0.8052, std=0.1064
RSF death     : mean=0.9434, std=0.0156
XGB hepatic   : mean=0.7591, std=0.1067
XGB death     : mean=0.9229, std=0.0390


## Summary

Prints the winning hyperparameters and blend weights found above for both endpoints, then re-runs the blended model's cross validated C index with the final config to report the competition metric: `0.7 * C index(hepatic) + 0.3 * C index(death)`.

In [18]:
print("=" * 60)
print("Hepatic")
print("=" * 60)
print(f"XGB hyperparams : {hep_xgb_params}")
print(f"Coxnet l1_ratio : {hep_l1_ratio}")
print(f"Blend weights   : (w_cox, w_rsf, w_xgb) = {hep_weights}")

print()
print("=" * 60)
print("Death")
print("=" * 60)
print(f"XGB hyperparams : {death_xgb_params}")
print(f"Coxnet l1_ratio : {death_l1_ratio}")
print(f"Blend weights   : (w_cox, w_rsf, w_xgb) = {death_weights}")

hep_blend_new = cv_cindex_blend(X_hep, y_hep, hep_event, hep_time, n_repeats=3, weights=config.blend_weights_hep(), xgb_params=config.xgb_hyperparams_hep())
death_blend_new = cv_cindex_blend(X_death, y_death, death_event, death_time, n_repeats=3, weights=config.blend_weights_death(), xgb_params=config.xgb_hyperparams_death())

weighted_new = 0.7 * hep_blend_new[0] + 0.3 * death_blend_new[0]
print(f"Hepatic: {hep_blend_new[0]:.4f}, Death: {death_blend_new[0]:.4f}, Weighted: {weighted_new:.4f}")

Hepatic
XGB hyperparams : {'learning_rate': 0.025, 'max_depth': 3, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_lambda': 5.0, 'reg_alpha': 0.5}
Coxnet l1_ratio : 0.1
Blend weights   : (w_cox, w_rsf, w_xgb) = (0.0, 1.0, 0.0)

Death
XGB hyperparams : {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.7, 'colsample_bytree': 0.85, 'reg_lambda': 1.0, 'reg_alpha': 0.5}
Coxnet l1_ratio : 0.99
Blend weights   : (w_cox, w_rsf, w_xgb) = (0.4, 0.4, 0.2)
Hepatic: 0.8052, Death: 0.9511, Weighted: 0.8490
